# Sharding

probjax modules are sharded **ambiently**. There are no sharding constructor arguments and
no configuration objects: modules bake *logical axis names* into their parameters, and if a
mesh is active when you construct one, flax shards the parameters as they are created
(eager sharding, FLIP 4844).

So the whole API is: set a mesh, build the model inside it, put the batch on the devices.

This notebook fakes 8 devices on the CPU so it can actually run. The flag has to be set
before JAX is imported.

In [1]:
import os

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=8"

import jax
import jax.numpy as jnp
import optax
from flax import nnx
from jax.sharding import AxisType, NamedSharding, PartitionSpec as P

from probjax.nn import MLP, Transformer
from probjax.nn.sharding import BATCH, EMBED, HEADS, HIDDEN, constrain, default_rules

print(jax.devices())

[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7)]


## 1. The mesh

A mesh names the device grid. Two axes is the usual arrangement: `data` splits the batch,
`model` splits the parameters.

**Use `axis_types=Auto`.** `jax.make_mesh` defaults to `Explicit` axes, where JAX requires
every intermediate's sharding to be stated rather than inferred — and
`with_sharding_constraint` asserts instead of propagating. Auto axes let GSPMD push
shardings through the contractions for you, which is what model parallelism needs.

In [2]:
print("jax.make_mesh default axis types:", jax.make_mesh((2, 4), ("data", "model")).axis_types)

mesh = jax.make_mesh((2, 4), ("data", "model"), axis_types=(AxisType.Auto, AxisType.Auto))
print("what we want:                      ", mesh)

jax.make_mesh default axis types: (Explicit, Explicit)
what we want:                       Mesh('data': 2, 'model': 4, axis_types=(Auto, Auto))


## 2. Logical axes

Modules never mention `"data"` or `"model"`. They annotate parameters with logical names —
`BATCH`, `SEQ`, `EMBED`, `HIDDEN`, `HEADS`, `HEAD_DIM` — which are resolved to physical mesh
axes at construction time. Swapping the parallelism strategy is then a change of rules, not
a change of model code.

The default mapping sends the two "wide" axes to the model axis and leaves the rest
replicated:

In [3]:
for logical, physical in default_rules():
    print(f"{logical:<9} -> {physical}")

batch     -> data
hidden    -> model
heads     -> model
embed     -> None
head_dim  -> None
seq       -> None


## 3. `MLP`: megatron-style tensor parallelism

Build under the mesh and the kernels come out sharded. The pattern alternates:

- **column parallel** `P(None, "model")` — split the *output* features, so each device
  computes a slice of the hidden layer with no communication;
- **row parallel** `P("model", None)` — split the *input* features, so each device
  consumes its own slice and the results are summed with a single all-reduce.

Alternating them means one collective per pair of layers rather than one per layer.

In [4]:
with jax.set_mesh(mesh):
    mlp = MLP([16, 64, 64, 16], rngs=nnx.Rngs(0))

for i, layer in enumerate(mlp.layers):
    kernel = layer.kernel[...]
    style = "column" if kernel.sharding.spec[-1] == "model" else "row   "
    print(f"layer {i}  {str(kernel.shape):<10} {style} parallel  {kernel.sharding.spec}")

layer 0  (16, 64)   column parallel  P(None, 'model')
layer 1  (64, 64)   row    parallel  P('model', None)
layer 2  (64, 16)   column parallel  P(None, 'model')


In [5]:
print("hidden kernel of layer 0, split across the 4 'model' devices:")
jax.debug.visualize_array_sharding(mlp.layers[0].kernel[...])

hidden kernel of layer 0, split across the 4 'model' devices:


                                                                                
                                                                                
                                                                                
                                                                                
                                                                                
         C…                  C…                  C…                  C…         
         0…                  1…                  2…                  3…         
                                                                                
                                                                                
                                                                                
                                                                                
                                                                                

Note the last layer here is column parallel, so the MLP's *output* is model-sharded rather
than replicated. That is a consequence of the layer count, not a bug — with an even number
of layers the stack ends row-parallel and the output comes back replicated.

**Divisibility is not negotiable.** Any feature dimension mapped to the model axis must be
divisible by it. `MLP([16, 64, 64, 1])` on a 4-way model axis fails at the final layer,
because 1 output feature cannot be split 4 ways:

In [6]:
with jax.set_mesh(mesh):
    try:
        MLP([16, 64, 64, 1], rngs=nnx.Rngs(0))
    except Exception as error:
        print(type(error).__name__, "->", str(error).split("\n")[0][:160])

IndivisibleError -> Sharding NamedSharding(mesh=Mesh('data': 2, 'model': 4, axis_types=(Auto, Auto)), spec=P(None, 'model'), memory_kind=device) implies that array axis 1 is partit


## 4. `Transformer`: sharding the head axis

Attention parallelises along heads: each device owns whole heads, so the softmax needs no
communication at all and only the output projection has to reduce. `HEADS` maps to the
model axis, `EMBED` and `HEAD_DIM` stay replicated.

In [7]:
with jax.set_mesh(mesh):
    model = Transformer(model_dim=32, num_heads=4, num_layers=2, attn_size=8, rngs=nnx.Rngs(0))

attention = model.attention_blocks[0]
for name in ("query", "key", "value", "out"):
    kernel = getattr(attention, name).kernel[...]
    print(f"{name:<6} {str(kernel.shape):<14} {kernel.sharding.spec}")

query  (32, 4, 8)     P(None, 'model', None)
key    (32, 4, 8)     P(None, 'model', None)
value  (32, 4, 8)     P(None, 'model', None)
out    (4, 8, 32)     P('model', None, None)


## 5. Data parallelism and a training step

Put the batch on the `data` axis and everything else follows. Inside `nnx.jit`, GSPMD
propagates the shardings through the computation — no manual annotation of intermediates.

In [8]:
with jax.set_mesh(mesh):
    trained = MLP([16, 64, 64, 16], rngs=nnx.Rngs(0))
    optimizer = nnx.Optimizer(trained, optax.adam(1e-2), wrt=nnx.Param)

    inputs = jax.device_put(jnp.ones((32, 16)), NamedSharding(mesh, P("data", None)))
    targets = jax.device_put(jnp.zeros((32, 16)), NamedSharding(mesh, P("data", None)))

    @nnx.jit
    def train_step(model, optimizer, x, y):
        def loss_fn(m):
            return ((m(x) - y) ** 2).mean()

        loss, grads = nnx.value_and_grad(loss_fn)(model)
        optimizer.update(model, grads)
        return loss

    for step in range(3):
        print(f"step {step}  loss {float(train_step(trained, optimizer, inputs, targets)):.5f}")

step 0  loss 0.28795
step 1  loss 0.03681
step 2  loss 0.02853


In [9]:
print("batch, split across the 2 'data' devices:")
jax.debug.visualize_array_sharding(inputs)

batch, split across the 2 'data' devices:


            
            
CPU 0,1,2,3 
            
            
            
            
            
CPU 4,5,6,7 
            
            
            

### Sharding is not supposed to change the answer

Worth checking rather than assuming — a mis-specified mesh usually still runs, it just
computes something subtly different.

In [10]:
with jax.set_mesh(mesh):
    sharded_out = nnx.jit(lambda m, v: m(v))(mlp, inputs)

reference = MLP([16, 64, 64, 16], rngs=nnx.Rngs(0))(jnp.ones((32, 16)))
print("output sharding:", sharded_out.sharding.spec)
print("max |sharded - single device|:", float(jnp.abs(jax.device_get(sharded_out) - reference).max()))

output sharding: P('data', 'model')
max |sharded - single device|: 2.980232238769531e-07


## 6. Changing the strategy without touching the model

Because modules only speak logical names, a different `nnx.logical_axis_rules` gives a
different parallelism strategy for the same code. Here `EMBED` goes to the model axis and
`HIDDEN` is replicated — the mirror image of the default.

In [11]:
custom = (
    (BATCH, "data"),
    (HIDDEN, None),
    (HEADS, "model"),
    (EMBED, "model"),
)

with jax.set_mesh(mesh), nnx.logical_axis_rules(custom):
    remapped = MLP([16, 64, 64, 16], rngs=nnx.Rngs(0))

print("default rules, layer 0:", mlp.layers[0].kernel[...].sharding.spec)
print("custom rules,  layer 0:", remapped.layers[0].kernel[...].sharding.spec)

default rules, layer 0: P(None, 'model')
custom rules,  layer 0: P('model', None)


`nnx.get_partition_spec` reads the resolved physical specs back out, which is how you give
optimizer state the same layout as the parameters:

In [12]:
with jax.set_mesh(mesh):
    specs = nnx.get_partition_spec(nnx.state(mlp, nnx.Param))

for path, spec in nnx.to_flat_state(specs):
    print(f"{'.'.join(str(p) for p in path):<16} {spec.get_value()}")

layers.0.bias    P('model',)
layers.0.kernel  P(None, 'model')
layers.1.bias    P()
layers.1.kernel  P('model', None)
layers.2.bias    P('model',)
layers.2.kernel  P(None, 'model')


## 7. Activations: `constrain` and `replicate`

Parameters are annotated once at construction; *activations* are pinned as they flow, with
`constrain(x, BATCH, SEQ, EMBED)` (mentioned axes only, trailing dims left free) and
`replicate(x)` for layers that need the whole array on every device — the LRU scan and
spatial self-attention, for example.

You rarely call these yourself; probjax modules already do. Both are no-ops without an
active mesh, inside a per-shard body, and on vmap-batched tracers (a constraint written for
the unbatched signature would land on the wrong dimension once vmap has moved the axes).

In [13]:
with jax.set_mesh(mesh):
    pinned = constrain(jnp.ones((32, 16)), BATCH, EMBED)
    print("inside a mesh: ", pinned.sharding.spec)

print("no mesh active:", constrain(jnp.ones((32, 16)), BATCH, EMBED).sharding)

inside a mesh:  P('data', None)
no mesh active: SingleDeviceSharding(device=CpuDevice(id=0), memory_kind=device)


## Gotchas

- **Use Auto axis types.** `jax.make_mesh` gives you `Explicit` by default, where
  `with_sharding_constraint` asserts rather than propagates.
- **Divisibility.** Every feature dimension mapped to the model axis must be divisible by
  that axis's size.
- **No mesh, no annotations.** Without an active mesh `param_metadata` returns `{}`
  deliberately — flax 0.12 raises at variable creation if a sharding annotation exists with
  no mesh to resolve it against.
- **Pallas attention under model parallelism is unvalidated.** Combining head-sharded
  attention parameters with `flex_attention` has not been checked; stay on the default
  attention path when sharding a `Transformer`.
- **Fake devices for testing.** `XLA_FLAGS=--xla_force_host_platform_device_count=8`
  reproduces this notebook's setup; the mesh test suite runs the same way
  (`pytest -m mesh`).

## Where to go next

- `transformer.ipynb` — the model being sharded here.
- `probjax/nn/sharding.py` — ~170 lines, and the authoritative reference.